In [7]:
# Visiting Card Scanner — Google Colab Edition
#
# This is a Google Colab-compatible replacement for the original Tkinter
# desktop application. It runs a Flask web application inside the notebook.
#
# Features:
# - Upload one or multiple visiting-card images
# - OCR with Tesseract
# - Extract name, designation, email, phone, and address heuristically
# - Save/update an Excel database
# - Export CSV and PDF files
# - Download saved OCR text and generated files
# - Run the application in the Colab output area
#
# Usage in Google Colab:
# 1. Paste this entire file into one Colab cell, or upload and run it.
# 2. Execute the cell.
# 3. Use the application shown below the cell.

from __future__ import annotations

import csv
import importlib.util
import os
import re
import shutil
import subprocess
import sys
import threading
import time
from functools import lru_cache
from pathlib import Path
from typing import Any


# -----------------------------------------------------------------------------
# One-time dependency setup for Google Colab
# -----------------------------------------------------------------------------


def install_if_missing(package_name: str, import_name: str | None = None) -> None:
    """Install a Python package only when it is not already available."""
    module_name = import_name or package_name
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])


# Colab normally includes these packages. The checks make the code reusable.
for _package, _module in [
    ("flask", "flask"),
    ("pillow", "PIL"),
    ("pytesseract", "pytesseract"),
    ("opencv-python-headless", "cv2"),
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("openpyxl", "openpyxl"),
    ("reportlab", "reportlab"),
]:
    install_if_missing(_package, _module)

# Tesseract is an operating-system program, not a Python package.
@lru_cache(maxsize=1)
def ensure_tesseract() -> bool:
    """Ensure Tesseract is available and return whether it can be executed."""
    import pytesseract

    candidates = [
        os.environ.get("TESSERACT_CMD", ""),
        "/usr/bin/tesseract",
        "/usr/local/bin/tesseract",
    ]
    for candidate in candidates:
        if candidate and Path(candidate).exists():
            pytesseract.pytesseract.tesseract_cmd = candidate
            try:
                pytesseract.get_tesseract_version()
                return True
            except Exception:
                pass

    if shutil.which("tesseract"):
        try:
            pytesseract.get_tesseract_version()
            return True
        except Exception:
            pass

    # Google Colab supports apt-get. This is skipped if installation is not
    # available, and the UI will display a useful OCR error instead.
    try:
        subprocess.run(
            ["apt-get", "update", "-qq"],
            check=True,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            timeout=180,
        )
        subprocess.run(
            ["apt-get", "install", "-y", "-qq", "tesseract-ocr"],
            check=True,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            timeout=180,
        )
        pytesseract.pytesseract.tesseract_cmd = shutil.which("tesseract") or "/usr/bin/tesseract"
        pytesseract.get_tesseract_version()
        return True
    except Exception:
        return False


# -----------------------------------------------------------------------------
# Imports after dependency setup
# -----------------------------------------------------------------------------

import pandas as pd
import cv2
import numpy as np
import pytesseract
from flask import Flask, jsonify, redirect, render_template_string, request, send_file, url_for
from PIL import Image, ImageEnhance, ImageFilter, ImageOps
from werkzeug.utils import secure_filename
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas


# -----------------------------------------------------------------------------
# Application folders and in-memory database
# -----------------------------------------------------------------------------

ROOT_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
DB_DIR = ROOT_DIR / "Database"
UPLOAD_DIR = DB_DIR / "uploads"
TEXT_DIR = DB_DIR / "ocr_text"
EXPORT_DIR = DB_DIR / "exports"
for _folder in (DB_DIR, UPLOAD_DIR, TEXT_DIR, EXPORT_DIR):
    _folder.mkdir(parents=True, exist_ok=True)

EXCEL_PATH = DB_DIR / "cards.xlsx"
CSV_PATH = EXPORT_DIR / "cards.csv"
PDF_PATH = EXPORT_DIR / "cards_report.pdf"
ALLOWED_EXTENSIONS = {"jpg", "jpeg", "png", "webp", "bmp", "tif", "tiff"}
DB_LOCK = threading.RLock()


def empty_database() -> pd.DataFrame:
    return pd.DataFrame(
        columns=["name", "designation", "email", "phone", "address", "filename", "text"]
    )


def load_database() -> pd.DataFrame:
    if not EXCEL_PATH.exists():
        return empty_database()
    try:
        df = pd.read_excel(EXCEL_PATH, engine="openpyxl")
        for column in empty_database().columns:
            if column not in df.columns:
                df[column] = ""
        return df[empty_database().columns].fillna("")
    except Exception:
        return empty_database()


DB_DF = load_database()


# -----------------------------------------------------------------------------
# OCR and card-detail extraction
# -----------------------------------------------------------------------------


def allowed_file(filename: str) -> bool:
    return "." in filename and filename.rsplit(".", 1)[1].lower() in ALLOWED_EXTENSIONS


def _order_points(points: np.ndarray) -> np.ndarray:
    points = np.asarray(points, dtype=np.float32)
    total = points.sum(axis=1)
    diff = np.diff(points, axis=1).ravel()
    return np.array(
        [points[np.argmin(total)], points[np.argmin(diff)],
         points[np.argmax(total)], points[np.argmax(diff)]], dtype=np.float32
    )


def _correct_card_perspective(image: np.ndarray) -> np.ndarray:
    """Flatten a card photographed at an angle when a clear rectangle exists."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(cv2.GaussianBlur(gray, (5, 5), 0), 50, 150)
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    image_area = image.shape[0] * image.shape[1]
    for contour in sorted(contours, key=cv2.contourArea, reverse=True)[:12]:
        area = cv2.contourArea(contour)
        perimeter = cv2.arcLength(contour, True)
        polygon = cv2.approxPolyDP(contour, 0.03 * perimeter, True)
        if len(polygon) != 4 or area < image_area * 0.20:
            continue
        corners = _order_points(polygon.reshape(4, 2))
        width = int(max(np.linalg.norm(corners[2] - corners[3]), np.linalg.norm(corners[1] - corners[0])))
        height = int(max(np.linalg.norm(corners[1] - corners[2]), np.linalg.norm(corners[0] - corners[3])))
        if width < 500 or height < 250:
            continue
        target = np.array([[0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1]], dtype=np.float32)
        matrix = cv2.getPerspectiveTransform(corners, target)
        return cv2.warpPerspective(image, matrix, (width, height))
    return image


def _deskew(image: np.ndarray) -> np.ndarray:
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    points = np.column_stack(np.where(binary > 0))
    if len(points) < 100:
        return image
    angle = cv2.minAreaRect(points.astype(np.float32))[-1]
    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle
    if abs(angle) < 0.5 or abs(angle) > 12:
        return image
    height, width = image.shape[:2]
    matrix = cv2.getRotationMatrix2D((width / 2, height / 2), angle, 1.0)
    return cv2.warpAffine(image, matrix, (width, height), borderMode=cv2.BORDER_REPLICATE)


def _ocr_quality(value: str) -> tuple[int, int, int, int]:
    lines = [line.strip() for line in value.splitlines() if line.strip()]
    contact_hits = len(re.findall(r"@|\+?\d[\d \t().-]{6,}\d", value))
    useful_chars = sum(char.isalnum() for char in value)
    return (contact_hits, len(lines), useful_chars, len(value))


@lru_cache(maxsize=1)
def _easy_reader():
    """Load EasyOCR only when it is available; Tesseract remains the fallback."""
    try:
        if importlib.util.find_spec("easyocr") is None:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "easyocr"])
        import easyocr
        return easyocr.Reader(["en"], gpu=False, verbose=False)
    except Exception:
        return None


def ocr_image(image_path: Path) -> str:
    """Flatten, deskew, enhance, and OCR a card using several independent passes."""
    if not ensure_tesseract():
        raise RuntimeError("Tesseract OCR could not be installed or started in this Colab runtime.")

    bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if bgr is None:
        raise ValueError("The uploaded file is not a readable image.")
    max_side = max(bgr.shape[:2])
    if max_side < 1800:
        scale = min(3.0, 1800 / max_side)
        bgr = cv2.resize(bgr, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    elif max_side > 3200:
        scale = 3200 / max_side
        bgr = cv2.resize(bgr, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)

    card = _deskew(_correct_card_perspective(bgr))
    gray = cv2.cvtColor(card, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8)).apply(gray)
    denoised = cv2.fastNlMeansDenoising(clahe, None, 7, 7, 21)
    adaptive = cv2.adaptiveThreshold(denoised, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 11)
    otsu = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
    variants = [card, gray, clahe, denoised, adaptive, otsu]
    candidates: list[str] = []
    for variant in variants:
        for psm in (6, 11, 12):
            result = pytesseract.image_to_string(variant, config=f"--oem 3 --psm {psm}").strip()
            if result:
                candidates.append(result)

    # EasyOCR often succeeds on stylized logos and text that Tesseract misses.
    reader = _easy_reader()
    if reader is not None:
        try:
            easy_lines = reader.readtext(card, detail=0, paragraph=False)
            easy_text = "\n".join(str(line).strip() for line in easy_lines if str(line).strip())
            if easy_text:
                candidates.append(easy_text)
        except Exception:
            pass

    # Choose the candidate containing the most useful lines and contact data,
    # while retaining all detected text in the displayed OCR result.
    return max(candidates, key=_ocr_quality, default="")


def parse_details(text: str) -> dict[str, str]:
    """Extract common visiting-card fields using conservative heuristics."""
    normalized = text.replace("\u2013", "-").replace("\u2014", "-")
    email_matches = re.findall(r"[\w.+'-]+@[\w.-]+\.[A-Za-z]{2,}", normalized)
    # Do not allow a phone match to cross an OCR line break and consume the
    # first digits of an address on the next line.
    phone_matches = re.findall(r"(?:\+?\d[\d \t().-]{6,}\d)", normalized)
    emails = list(dict.fromkeys(email_matches))
    phones = list(dict.fromkeys(" ".join(phone.split()) for phone in phone_matches))

    lines = [line.strip(" |,;") for line in normalized.splitlines() if line.strip()]
    # Business-card typography often places first and last names on separate
    # lines. Join two short alphabetic heading lines before field detection.
    if len(lines) >= 2:
        first, second = lines[0], lines[1]
        if (
            re.fullmatch(r"[A-Za-z][A-Za-z .'-]{1,30}", first)
            and re.fullmatch(r"[A-Za-z][A-Za-z .'-]{1,30}", second)
            and len(first.split()) <= 3
            and len(second.split()) <= 3
            and first.upper() == first
            and second.upper() == second
        ):
            lines = [f"{first} {second}"] + lines[2:]
    name = ""
    designation = ""
    address_lines: list[str] = []

    # The first line with letters and no contact indicators is a reasonable
    # name candidate. OCR results are heuristic and should be reviewed.
    designation_words = (
        "ceo", "cto", "cfo", "manager", "director", "engineer", "developer",
        "president", "founder", "officer", "consultant", "designer", "professor",
        "student", "owner", "lead", "sales", "marketing", "hr", "accountant",
    )
    for index, line in enumerate(lines):
        lower = line.lower()
        if any(token in lower for token in ("www.", "http://", "https://", "@")):
            continue
        if re.search(r"\d", line):
            continue
        if re.search(r"[A-Za-z]", line):
            name = line
            for next_line in lines[index + 1 :]:
                next_lower = next_line.lower()
                if "@" in next_line or "www." in next_lower or re.search(r"\d", next_line):
                    continue
                if any(word in next_line.lower() for word in designation_words):
                    designation = next_line
                    break
                if not designation:
                    designation = next_line
                    break
            break

    if name:
        start_index = lines.index(designation) + 1 if designation in lines else lines.index(name) + 1
    else:
        start_index = 0

    for line in lines[start_index:]:
        lower = line.lower()
        if "@" in line or "www." in lower or "http" in lower:
            continue
        if re.search(r"\d{5,}", line):
            continue
        if line in {name, designation}:
            continue
        address_lines.append(line)

    return {
        "name": name,
        "designation": designation,
        "email": emails[0] if emails else "",
        "phone": phones[0] if phones else "",
        "address": " | ".join(address_lines),
        "raw": text,
    }


def save_text_file(filename: str, text: str) -> Path:
    safe_name = secure_filename(Path(filename).stem) or "card"
    text_path = TEXT_DIR / f"{safe_name}.txt"
    text_path.write_text(text, encoding="utf-8")
    return text_path


def record_from_text(text: str, filename: str) -> dict[str, str]:
    details = parse_details(text)
    return {
        "name": details["name"],
        "designation": details["designation"],
        "email": details["email"],
        "phone": details["phone"],
        "address": details["address"],
        "filename": filename,
        "text": text,
    }


def save_record(record: dict[str, str]) -> None:
    global DB_DF
    with DB_LOCK:
        DB_DF = pd.concat([DB_DF, pd.DataFrame([record])], ignore_index=True)
        DB_DF.to_excel(EXCEL_PATH, index=False, engine="openpyxl")


# -----------------------------------------------------------------------------
# Flask application and routes
# -----------------------------------------------------------------------------

app = Flask(__name__)
app.config["MAX_CONTENT_LENGTH"] = 25 * 1024 * 1024


@app.get("/")
def home():
    return render_template_string(
        PAGE,
        rows=DB_DF.to_dict("records"),
        message=request.args.get("message", ""),
        results=[],
    )


@app.post("/scan")
def scan():
    uploaded = request.files.getlist("images")
    if not uploaded or not any(item.filename for item in uploaded):
        return redirect(url_for("home", message="Please select at least one image."))

    results = []
    errors = []
    for item in uploaded:
        if not item.filename or not allowed_file(item.filename):
            errors.append(f"Skipped unsupported file: {item.filename}")
            continue
        safe_name = secure_filename(item.filename)
        image_path = UPLOAD_DIR / safe_name
        item.save(image_path)
        try:
            text = ocr_image(image_path)
            if not text:
                errors.append(f"No text detected in {safe_name}.")
                continue
            save_text_file(safe_name, text)
            record = record_from_text(text, safe_name)
            save_record(record)
            results.append(record)
        except Exception as exc:
            errors.append(f"Could not process {safe_name}: {exc}")

    message = f"Processed {len(results)} image(s)."
    if errors:
        message += " " + " ".join(errors)
    return render_template_string(PAGE, rows=DB_DF.to_dict("records"), message=message, results=results)


@app.post("/save-text")
def save_text():
    filename = request.form.get("filename", "manual_card.txt")
    text = request.form.get("text", "").strip()
    if not text:
        return redirect(url_for("home", message="Enter OCR text before saving."))
    path = save_text_file(filename, text)
    save_record(record_from_text(text, Path(filename).name))
    return redirect(url_for("home", message=f"Saved OCR text to {path.name} and updated Excel."))


@app.get("/download/csv")
def download_csv():
    with DB_LOCK:
        DB_DF.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
    return send_file(CSV_PATH, as_attachment=True, download_name="cards.csv")


@app.get("/download/excel")
def download_excel():
    if not EXCEL_PATH.exists():
        DB_DF.to_excel(EXCEL_PATH, index=False, engine="openpyxl")
    return send_file(EXCEL_PATH, as_attachment=True, download_name="cards.xlsx")


@app.get("/download/pdf")
def download_pdf():
    if DB_DF.empty:
        return redirect(url_for("home", message="No records are available for PDF export."))

    c = canvas.Canvas(str(PDF_PATH), pagesize=letter)
    width, height = letter
    y = height - 45
    c.setTitle("Visiting Card Scanner Report")
    c.setFont("Helvetica-Bold", 14)
    c.drawString(45, y, "Visiting Card Scanner Report")
    y -= 28
    c.setFont("Helvetica", 9)

    for index, row in DB_DF.iterrows():
        fields = [
            f"Record {index + 1}",
            f"Name: {row.get('name', '')}",
            f"Designation: {row.get('designation', '')}",
            f"Email: {row.get('email', '')}",
            f"Phone: {row.get('phone', '')}",
            f"Address: {row.get('address', '')}",
            f"File: {row.get('filename', '')}",
        ]
        for field in fields:
            if y < 55:
                c.showPage()
                y = height - 45
                c.setFont("Helvetica", 9)
            c.drawString(45, y, str(field)[:125])
            y -= 13
        y -= 8
    c.save()
    return send_file(PDF_PATH, as_attachment=True, download_name="cards_report.pdf")


@app.get("/download/text/<path:filename>")
def download_text(filename: str):
    safe_name = secure_filename(filename)
    path = TEXT_DIR / safe_name
    if not path.exists():
        return jsonify(error="Text file not found."), 404
    return send_file(path, as_attachment=True, download_name=path.name)


# -----------------------------------------------------------------------------
# HTML user interface
# -----------------------------------------------------------------------------

PAGE = r"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Visiting Card Scanner — Google Colab</title>
<style>
*{box-sizing:border-box}body{margin:0;background:#f5f7fb;color:#202938;font-family:Arial,sans-serif}
.wrap{max-width:1120px;margin:24px auto;padding:0 16px 32px}.hero{background:#f5c7a8;border-radius:14px;padding:22px 24px;margin-bottom:16px}
h1{margin:0 0 4px;font-size:28px}.hero p{margin:0;color:#5b4438}.grid{display:grid;grid-template-columns:1fr 1fr;gap:16px}
.card{background:#fff;border:1px solid #dde3ed;border-radius:14px;padding:18px;box-shadow:0 4px 18px #15284a0d}h2{margin:0 0 14px;font-size:19px}
input[type=file],input[type=text],textarea{width:100%;padding:10px;border:1px solid #cbd5e1;border-radius:8px;font:inherit}textarea{min-height:150px;resize:vertical}
button,.btn{display:inline-block;border:0;border-radius:8px;padding:10px 14px;background:#f58d4b;color:#24160d;font-weight:700;text-decoration:none;cursor:pointer;margin:6px 4px 0 0}
.btn-blue{background:#8ec5ff}.btn-green{background:#8eff9f}.btn-yellow{background:#ffd98e}.btn-purple{background:#c28eff}.btn-gray{background:#e4e7eb}.btn-red{background:#ff8e8e}
.notice{background:#edf7ee;border-left:4px solid #2e9d59;padding:10px 12px;border-radius:6px;margin-bottom:14px;white-space:pre-wrap}.warn{background:#fff5df;border-left-color:#df9816}
.table-wrap{overflow:auto;max-height:420px}table{border-collapse:collapse;width:100%;font-size:13px}th,td{text-align:left;padding:8px;border-bottom:1px solid #edf0f5;vertical-align:top}th{color:#5d6b80;white-space:nowrap}
pre{white-space:pre-wrap;word-break:break-word;background:#f7f8fa;padding:12px;border-radius:8px;min-height:100px}.download-row{display:flex;flex-wrap:wrap;gap:4px}
@media(max-width:800px){.grid{grid-template-columns:1fr}}
</style>
</head>
<body><main class="wrap">
<header class="hero"><h1>Visiting Card Scanner</h1><p>OCR, detail extraction, Excel database, CSV export, and PDF reporting — running in Google Colab.</p></header>
{% if message %}<div class="notice {% if 'Could not' in message or 'Please' in message %}warn{% endif %}">{{ message }}</div>{% endif %}
<div class="grid">
<section class="card"><h2>Upload Card Images</h2>
<form action="/scan" method="post" enctype="multipart/form-data">
<input type="file" name="images" accept="image/*" multiple required>
<button type="submit">Scan and Convert</button></form>
<p><small>Select one or more JPG, PNG, WEBP, BMP, or TIFF files. OCR results are saved in the Database folder.</small></p>
<h2 style="margin-top:22px">Save OCR Text Manually</h2>
<form action="/save-text" method="post"><input type="text" name="filename" placeholder="card_filename.txt" value="manual_card.txt"><br><textarea name="text" placeholder="Paste converted card text here..."></textarea><br><button class="btn-green" type="submit">Save / Update Excel</button></form>
</section>
<section class="card"><h2>Exports</h2><div class="download-row"><a class="btn btn-yellow" href="/download/csv">Download CSV</a><a class="btn btn-green" href="/download/excel">Download Excel</a><a class="btn btn-purple" href="/download/pdf">Download PDF Report</a></div>
<h2 style="margin-top:24px">Saved Card Records ({{ rows|length }})</h2>
<div class="table-wrap"><table><thead><tr><th>Name</th><th>Designation</th><th>Email</th><th>Phone</th><th>Address</th><th>File</th></tr></thead><tbody>
{% for row in rows %}<tr><td>{{ row.name }}</td><td>{{ row.designation }}</td><td>{{ row.email }}</td><td>{{ row.phone }}</td><td>{{ row.address }}</td><td>{{ row.filename }}</td></tr>{% else %}<tr><td colspan="6">No records yet.</td></tr>{% endfor %}
</tbody></table></div></section>
</div>
{% if results %}<section class="card" style="margin-top:16px"><h2>Latest OCR Results</h2>{% for result in results %}<h3>{{ result.filename }}</h3><p><b>Name:</b> {{ result.name }} &nbsp; <b>Designation:</b> {{ result.designation }}<br><b>Email:</b> {{ result.email }} &nbsp; <b>Phone:</b> {{ result.phone }}<br><b>Address:</b> {{ result.address }}</p><pre>{{ result.text }}</pre>{% endfor %}</section>{% endif %}
</main></body></html>"""


# -----------------------------------------------------------------------------
# Google Colab launcher
# -----------------------------------------------------------------------------


def start_colab_app(port: int = 5000):
    """Start Flask in the background and show it in the Colab output area."""
    thread = threading.Thread(
        target=lambda: app.run(host="0.0.0.0", port=port, debug=False, use_reloader=False),
        daemon=True,
    )
    thread.start()
    time.sleep(2)
    print(f"Visiting Card Scanner is running on port {port}.")

    try:
        from google.colab.output import serve_kernel_port_as_iframe
        serve_kernel_port_as_iframe(port, height=900)
    except ImportError:
        print(f"Open http://127.0.0.1:{port}/ in a normal Python environment.")

    return thread


# Run automatically when this file is executed in Google Colab.
if __name__ == "__main__":
    start_colab_app()


 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


Visiting Card Scanner is running on port 5000.


<IPython.core.display.Javascript object>